# LLaMA Factory를 이용한 QLoRA 파인튜닝 실습

## 📚 강의 개요

이 노트북에서는 **LLaMA Factory**를 사용하여 의료 도메인 데이터셋으로 sLLM을 QLoRA 방식으로 파인튜닝하는 전체 과정을 학습합니다.

### 학습 목표
- ✅ LLaMA Factory의 핵심 개념 이해
- ✅ QLoRA를 이용한 효율적인 파인튜닝 방법 습득
- ✅ 의료 도메인 데이터셋 전처리 및 포맷팅
- ✅ 모델 학습, 평가 및 배포 전 과정 실습

### 실습 환경
- **프레임워크**: LLaMA Factory 0.8.3
- **학습 방법**: QLoRA (4-bit 양자화)
- **데이터셋**: 소아청소년과 의료 상담 데이터 (795 samples)
- **모델**: Qwen 2.5 1.5B (경량 모델)


---

## 1️⃣ 환경 설정 및 라이브러리 확인

먼저 설치된 라이브러리들을 확인하고, 필요한 패키지들을 임포트합니다.


In [ ]:
# 주요 라이브러리 버전 확인
import sys
import torch
import transformers
import peft
import trl
import datasets

print("=" * 60)
print("환경 정보")
print("=" * 60)
print(f"Python 버전: {sys.version.split()[0]}")
print(f"PyTorch 버전: {torch.__version__}")
print(f"Transformers 버전: {transformers.__version__}")
print(f"PEFT 버전: {peft.__version__}")
print(f"TRL 버전: {trl.__version__}")
print(f"Datasets 버전: {datasets.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA 버전: {torch.version.cuda}")
    print(f"GPU 개수: {torch.cuda.device_count()}")
    print(f"현재 GPU: {torch.cuda.get_device_name(0)}")
print("=" * 60)


In [ ]:
# 필요한 라이브러리 임포트
import os
import json
import pandas as pd
from pathlib import Path
from typing import Dict, List, Optional

# 경고 메시지 숨기기
import warnings
warnings.filterwarnings('ignore')

# 환경 변수 설정
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("✅ 라이브러리 임포트 완료!")


---

## 2️⃣ 데이터셋 탐색 및 분석

의료 상담 데이터셋을 로드하고 구조를 파악합니다.


In [ ]:
# 데이터셋 로드
data_path = "data/illnesses.csv"
df = pd.read_csv(data_path)

print("=" * 60)
print("데이터셋 기본 정보")
print("=" * 60)
print(f"전체 샘플 수: {len(df):,}")
print(f"컬럼 수: {len(df.columns)}")
print(f"\n컬럼 목록:")
for col in df.columns:
    print(f"  - {col}")
print("=" * 60)

# 처음 3개 샘플 확인
df.head(3)


In [ ]:
# 데이터 분석: Query Style 분포
print("=" * 60)
print("Query Style 분포")
print("=" * 60)
style_counts = df['query_style'].value_counts()
for style, count in style_counts.items():
    percentage = (count / len(df)) * 100
    print(f"{style:20s}: {count:4d} ({percentage:5.1f}%)")
print("=" * 60)

# Query Length 분포
print("\nQuery Length 분포")
print("=" * 60)
length_counts = df['query_length'].value_counts()
for length, count in length_counts.items():
    percentage = (count / len(df)) * 100
    print(f"{length:20s}: {count:4d} ({percentage:5.1f}%)")
print("=" * 60)


In [ ]:
# 샘플 데이터 출력 (가독성 있게)
print("=" * 60)
print("샘플 데이터 (첫 번째)")
print("=" * 60)

sample = df.iloc[0]
print(f"📝 질문 (User Input):")
print(f"   {sample['user_input']}\n")
print(f"✅ 답변 (Reference):")
print(f"   {sample['reference']}\n")
print(f"👤 페르소나: {sample['persona_name']}")
print(f"📊 질문 스타일: {sample['query_style']}")
print(f"📏 질문 길이: {sample['query_length']}")
print(f"🎥 참고 영상: {sample['reference_video_url']}")
print("=" * 60)


---

## 3️⃣ LLaMA Factory용 데이터셋 포맷팅

LLaMA Factory는 특정 JSON 포맷을 요구합니다. 데이터를 변환하고 저장합니다.

### LLaMA Factory 데이터셋 포맷

```json
[
  {
    "instruction": "시스템 프롬프트 (선택사항)",
    "input": "사용자 질문",
    "output": "모델 답변"
  }
]
```


In [ ]:
# LLaMA Factory 포맷으로 변환
def convert_to_llamafactory_format(df: pd.DataFrame) -> List[Dict]:
    """
    CSV 데이터를 LLaMA Factory 포맷으로 변환
    """
    data_list = []
    
    # 시스템 프롬프트 정의
    system_instruction = (
        "당신은 소아청소년과 전문의입니다. "
        "부모님들의 질문에 전문적이고 정확하며 이해하기 쉬운 답변을 제공합니다. "
        "의학적 근거를 바탕으로 신뢰할 수 있는 조언을 해주세요."
    )
    
    for _, row in df.iterrows():
        data_item = {
            "instruction": system_instruction,
            "input": row['user_input'],
            "output": row['reference']
        }
        data_list.append(data_item)
    
    return data_list

# 데이터 변환
formatted_data = convert_to_llamafactory_format(df)

print(f"✅ 총 {len(formatted_data):,}개 샘플 변환 완료!")
print(f"\n첫 번째 샘플 예시:")
print(json.dumps(formatted_data[0], ensure_ascii=False, indent=2)[:500] + "...")


In [ ]:
# Train/Validation 분할 (90% / 10%)
from sklearn.model_selection import train_test_split

train_data, val_data = train_test_split(
    formatted_data, 
    test_size=0.1, 
    random_state=42
)

print("=" * 60)
print("데이터셋 분할 완료")
print("=" * 60)
print(f"전체 데이터: {len(formatted_data):,} 샘플")
print(f"학습 데이터: {len(train_data):,} 샘플 ({len(train_data)/len(formatted_data)*100:.1f}%)")
print(f"검증 데이터: {len(val_data):,} 샘플 ({len(val_data)/len(formatted_data)*100:.1f}%)")
print("=" * 60)


In [ ]:
# JSON 파일로 저장
output_dir = Path("data/llamafactory")
output_dir.mkdir(exist_ok=True)

train_path = output_dir / "train.json"
val_path = output_dir / "val.json"

with open(train_path, 'w', encoding='utf-8') as f:
    json.dump(train_data, f, ensure_ascii=False, indent=2)

with open(val_path, 'w', encoding='utf-8') as f:
    json.dump(val_data, f, ensure_ascii=False, indent=2)

print(f"✅ 데이터셋 저장 완료!")
print(f"   - 학습 데이터: {train_path}")
print(f"   - 검증 데이터: {val_path}")


---

## 4️⃣ 모델 및 토크나이저 로드

경량화된 **Qwen 2.5 1.5B** 모델을 사용합니다. 4-bit 양자화를 적용하여 메모리를 절약합니다.

### QLoRA 핵심 개념

- **4-bit 양자화**: 모델 가중치를 4비트로 압축 (메모리 75% 감소)
- **LoRA (Low-Rank Adaptation)**: 소수의 파라미터만 학습
- **Gradient Checkpointing**: 메모리 효율적인 역전파


In [ ]:
# 모델 및 토크나이저 설정
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

# 모델 선택 (Qwen 2.5 1.5B - 경량 모델)
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"🤖 모델 로드 중: {MODEL_NAME}")
print(f"   이 작업은 몇 분 정도 소요될 수 있습니다...")


In [ ]:
# 4-bit 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                      # 4-bit 양자화 활성화
    bnb_4bit_use_double_quant=True,         # 이중 양자화 (메모리 추가 절약)
    bnb_4bit_quant_type="nf4",              # NormalFloat4 양자화 타입
    bnb_4bit_compute_dtype=torch.bfloat16  # 연산은 bfloat16으로
)

print("=" * 60)
print("4-bit 양자화 설정")
print("=" * 60)
print(f"✅ 4-bit 로딩: 활성화")
print(f"✅ 이중 양자화: 활성화 (메모리 추가 절약)")
print(f"✅ 양자화 타입: NF4 (NormalFloat4)")
print(f"✅ 연산 dtype: bfloat16")
print("=" * 60)


In [ ]:
# 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

# padding token 설정 (없으면 eos_token 사용)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✅ 토크나이저 로드 완료!")
print(f"   - Vocab 크기: {len(tokenizer):,}")
print(f"   - PAD token: {tokenizer.pad_token}")
print(f"   - EOS token: {tokenizer.eos_token}")


In [ ]:
# 모델 로드 (4-bit 양자화 적용)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",              # 자동으로 GPU에 배치
    trust_remote_code=True,
    torch_dtype=torch.bfloat16
)

print(f"✅ 모델 로드 완료!")

# 모델 정보 출력
total_params = sum(p.numel() for p in model.parameters())
print(f"\n모델 정보:")
print(f"   - 전체 파라미터: {total_params:,}")


---

## 5️⃣ LoRA 설정 및 적용

LoRA를 통해 학습 가능한 파라미터를 대폭 줄입니다.

### LoRA 하이퍼파라미터

- **r (rank)**: LoRA 행렬의 rank (낮을수록 파라미터 적음)
- **alpha**: 학습률 스케일링
- **dropout**: 과적합 방지


In [ ]:
# LoRA 설정
lora_config = LoraConfig(
    r=16,                          # LoRA rank (16, 32, 64 등)
    lora_alpha=32,                 # LoRA alpha (보통 r의 2배)
    target_modules=[               # LoRA를 적용할 모듈
        "q_proj",
        "k_proj", 
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,             # Dropout 비율
    bias="none",                   # bias 학습 안함
    task_type="CAUSAL_LM"          # Causal Language Modeling
)

print("=" * 60)
print("LoRA 설정")
print("=" * 60)
print(f"Rank (r): {lora_config.r}")
print(f"Alpha: {lora_config.lora_alpha}")
print(f"Dropout: {lora_config.lora_dropout}")
print(f"Target Modules: {', '.join(lora_config.target_modules)}")
print("=" * 60)


In [ ]:
# 모델을 k-bit 학습에 맞게 준비
model = prepare_model_for_kbit_training(model)

# LoRA 적용
model = get_peft_model(model, lora_config)

# 학습 가능한 파라미터 확인
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
trainable_percentage = (trainable_params / total_params) * 100

print("=" * 60)
print("LoRA 적용 후 모델 정보")
print("=" * 60)
print(f"전체 파라미터:     {total_params:,}")
print(f"학습 가능 파라미터: {trainable_params:,}")
print(f"학습 비율:         {trainable_percentage:.4f}%")
print(f"메모리 절약:       약 {100 - trainable_percentage:.2f}%")
print("=" * 60)


---

## 6️⃣ 데이터셋 전처리

학습 데이터를 모델이 이해할 수 있는 형태로 변환합니다.


In [ ]:
# Dataset 객체 생성
from datasets import Dataset

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

print("✅ Dataset 객체 생성 완료!")
print(f"   - 학습 데이터: {len(train_dataset):,} 샘플")
print(f"   - 검증 데이터: {len(val_dataset):,} 샘플")
print(f"\n데이터셋 컬럼: {train_dataset.column_names}")


In [ ]:
# 프롬프트 포맷팅 함수
def format_prompt(sample: Dict) -> str:
    """
    Qwen 모델의 프롬프트 형식으로 변환
    """
    return f"""<|im_start|>system
{sample['instruction']}<|im_end|>
<|im_start|>user
{sample['input']}<|im_end|>
<|im_start|>assistant
{sample['output']}<|im_end|>"""

# 샘플 프롬프트 확인
sample_prompt = format_prompt(train_data[0])
print("=" * 60)
print("프롬프트 포맷 예시")
print("=" * 60)
print(sample_prompt[:500] + "...")
print("=" * 60)


---

## 7️⃣ 학습 설정 및 실행

트레이너를 설정하고 모델 학습을 시작합니다.

### 주요 하이퍼파라미터

- **Learning Rate**: 학습률 (QLoRA는 보통 1e-4 ~ 2e-4)
- **Batch Size**: 배치 크기 (GPU 메모리에 따라 조절)
- **Epochs**: 에포크 수
- **Max Sequence Length**: 최대 시퀀스 길이


In [ ]:
# 학습 설정
output_dir = "./models/qwen-medical-qlora"

training_args = TrainingArguments(
    # 출력 디렉토리
    output_dir=output_dir,
    
    # 학습 하이퍼파라미터
    num_train_epochs=3,                    # 에포크 수
    per_device_train_batch_size=4,         # 학습 배치 크기
    per_device_eval_batch_size=4,          # 평가 배치 크기
    gradient_accumulation_steps=4,         # Gradient Accumulation (effective batch = 4*4=16)
    
    # 옵티마이저
    learning_rate=2e-4,                    # 학습률
    lr_scheduler_type="cosine",            # 학습률 스케줄러
    warmup_ratio=0.03,                     # Warmup 비율
    
    # 로깅 및 저장
    logging_steps=10,                      # 로그 출력 주기
    save_strategy="epoch",                 # 저장 전략
    evaluation_strategy="epoch",           # 평가 전략
    
    # 최적화
    fp16=False,                            # FP16 학습 (Ampere 이상 GPU)
    bf16=True,                             # BF16 학습 (권장)
    gradient_checkpointing=True,           # Gradient Checkpointing (메모리 절약)
    
    # 기타
    optim="paged_adamw_8bit",              # 8-bit AdamW 옵티마이저
    weight_decay=0.01,                     # Weight Decay
    max_grad_norm=0.3,                     # Gradient Clipping
    report_to="none",                      # TensorBoard/WandB 비활성화
    
    # 재현성
    seed=42,
)

print("=" * 60)
print("학습 설정")
print("=" * 60)
print(f"에포크 수: {training_args.num_train_epochs}")
print(f"배치 크기: {training_args.per_device_train_batch_size}")
print(f"Gradient Accumulation: {training_args.gradient_accumulation_steps}")
print(f"Effective Batch Size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"학습률: {training_args.learning_rate}")
print(f"옵티마이저: {training_args.optim}")
print(f"출력 디렉토리: {output_dir}")
print("=" * 60)


In [ ]:
# SFT Trainer 생성
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    formatting_func=format_prompt,         # 프롬프트 포맷팅 함수
    max_seq_length=1024,                   # 최대 시퀀스 길이
    packing=False,                         # 여러 샘플을 하나로 패킹 (메모리 효율)
)

print("✅ SFT Trainer 생성 완료!")


In [ ]:
# 학습 시작!
print("=" * 60)
print("🚀 학습 시작!")
print("=" * 60)
print("이 작업은 GPU 성능에 따라 10분~1시간 정도 소요될 수 있습니다.")
print("=" * 60)

trainer.train()


---

## 8️⃣ 학습 결과 분석

학습 로그를 통해 모델 성능을 확인합니다.


In [ ]:
# 학습 히스토리 확인
import matplotlib.pyplot as plt

history = trainer.state.log_history

# Train Loss 추출
train_logs = [log for log in history if 'loss' in log and 'eval_loss' not in log]
train_steps = [log['step'] for log in train_logs]
train_losses = [log['loss'] for log in train_logs]

# Eval Loss 추출
eval_logs = [log for log in history if 'eval_loss' in log]
eval_steps = [log['step'] for log in eval_logs]
eval_losses = [log['eval_loss'] for log in eval_logs]

# 시각화
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(train_steps, train_losses, 'b-', label='Train Loss')
plt.xlabel('Steps')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(eval_steps, eval_losses, 'r-', label='Eval Loss')
plt.xlabel('Steps')
plt.ylabel('Loss')
plt.title('Evaluation Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

print(f"✅ 최종 Train Loss: {train_losses[-1]:.4f}")
print(f"✅ 최종 Eval Loss: {eval_losses[-1]:.4f}")


In [ ]:
# 추론 함수
def generate_response(question: str, max_new_tokens: int = 512) -> str:
    """
    질문에 대한 답변 생성
    """
    system_instruction = (
        "당신은 소아청소년과 전문의입니다. "
        "부모님들의 질문에 전문적이고 정확하며 이해하기 쉬운 답변을 제공합니다. "
        "의학적 근거를 바탕으로 신뢰할 수 있는 조언을 해주세요."
    )
    
    prompt = f"""<|im_start|>system
{system_instruction}<|im_end|>
<|im_start|>user
{question}<|im_end|>
<|im_start|>assistant
"""
    
    # 토크나이징
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # 생성
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    # 디코딩
    response = tokenizer.decode(outputs[0], skip_special_tokens=False)
    
    # assistant 부분만 추출
    if "<|im_start|>assistant" in response:
        response = response.split("<|im_start|>assistant")[-1]
        response = response.replace("<|im_end|>", "").strip()
    
    return response

print("✅ 추론 함수 준비 완료!")


In [ ]:
# 테스트 질문들
test_questions = [
    "아이가 열이 39도인데 어떻게 해야 하나요?",
    "수족구병이 의심되는데 병원에 가야 하나요?",
    "독감 예방접종은 언제 맞추는 게 좋나요?",
]

print("=" * 60)
print("모델 테스트")
print("=" * 60)

for i, question in enumerate(test_questions, 1):
    print(f"\n[질문 {i}]")
    print(f"❓ {question}")
    print(f"\n[답변]")
    
    response = generate_response(question)
    print(f"💬 {response}")
    print("-" * 60)


---

## 🔟 모델 저장 및 내보내기

학습된 모델을 저장하고 다양한 형식으로 내보냅니다.


In [ ]:
# LoRA 어댑터만 저장
lora_output_dir = "./models/qwen-medical-lora"
model.save_pretrained(lora_output_dir)
tokenizer.save_pretrained(lora_output_dir)

print(f"✅ LoRA 어댑터 저장 완료!")
print(f"   위치: {lora_output_dir}")
print(f"\n저장된 파일:")
for file in Path(lora_output_dir).glob("*"):
    print(f"   - {file.name}")


In [ ]:
# 전체 모델 병합 (선택사항)
print("=" * 60)
print("모델 병합 (Base Model + LoRA)")
print("=" * 60)
print("베이스 모델과 LoRA 가중치를 병합하면 단일 모델로 사용 가능합니다.")
print("이 작업은 메모리를 많이 사용하므로 필요한 경우에만 수행하세요.")
print("=" * 60)

# 병합 코드 (주석 처리)
# from peft import PeftModel
# 
# # 베이스 모델 다시 로드 (FP16)
# base_model = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME,
#     torch_dtype=torch.float16,
#     device_map="auto"
# )
# 
# # LoRA 어댑터 로드 및 병합
# merged_model = PeftModel.from_pretrained(base_model, lora_output_dir)
# merged_model = merged_model.merge_and_unload()
# 
# # 병합된 모델 저장
# merged_output_dir = "./models/qwen-medical-merged"
# merged_model.save_pretrained(merged_output_dir)
# tokenizer.save_pretrained(merged_output_dir)
# 
# print(f"✅ 병합된 모델 저장 완료: {merged_output_dir}")


---

## 📊 요약 및 다음 단계

### 이번 실습에서 배운 것

✅ **LLaMA Factory 기본 사용법**
- 데이터셋 준비 및 포맷팅
- 모델 로드 및 설정
- QLoRA를 이용한 효율적 파인튜닝

✅ **QLoRA 핵심 개념**
- 4-bit 양자화로 메모리 75% 절약
- LoRA로 학습 파라미터 99% 이상 감소
- Gradient Checkpointing으로 추가 메모리 절약

✅ **의료 도메인 특화 모델 개발**
- 소아청소년과 데이터로 파인튜닝
- 전문적이고 신뢰할 수 있는 답변 생성
- 실무 적용 가능한 모델 개발

### 다음 단계

🔹 **모델 최적화**
- GGUF 형식으로 변환하여 추론 속도 개선
- llama.cpp로 CPU에서도 실행 가능하도록 변환

🔹 **성능 개선**
- 하이퍼파라미터 튜닝 (learning rate, batch size, LoRA rank 등)
- 더 많은 데이터로 학습
- DPO, PPO 등 고급 학습 방법 적용

🔹 **프로덕션 배포**
- FastAPI를 이용한 API 서버 구축
- Gradio/Streamlit으로 웹 UI 개발
- Docker 컨테이너화

🔹 **모델 평가**
- 자동 평가 메트릭 (BLEU, ROUGE 등)
- 전문가 평가
- A/B 테스트

### 참고 자료

- [LLaMA Factory GitHub](https://github.com/hiyouga/LLaMA-Factory)
- [QLoRA 논문](https://arxiv.org/abs/2305.14314)
- [PEFT 라이브러리](https://github.com/huggingface/peft)
- [TRL 라이브러리](https://github.com/huggingface/trl)

---

**🎉 수고하셨습니다! LLaMA Factory를 이용한 QLoRA 파인튜닝 실습을 완료했습니다!**


In [ ]:
# 저장된 LoRA 어댑터 로드하기
from peft import PeftModel, PeftConfig

# 방법 1: 베이스 모델 + LoRA 어댑터
def load_lora_model(base_model_name: str, lora_adapter_path: str):
    """
    저장된 LoRA 어댑터를 로드
    """
    # 베이스 모델 로드
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    
    # LoRA 어댑터 로드
    model = PeftModel.from_pretrained(base_model, lora_adapter_path)
    
    # 토크나이저 로드
    tokenizer = AutoTokenizer.from_pretrained(lora_adapter_path)
    
    return model, tokenizer

# 사용 예시
# model, tokenizer = load_lora_model(
#     base_model_name="Qwen/Qwen2.5-1.5B-Instruct",
#     lora_adapter_path="./models/qwen-medical-lora"
# )

print("✅ LoRA 어댑터 로드 함수 준비 완료!")
print("\n나중에 모델을 다시 사용하려면:")
print("1. 베이스 모델 로드")
print("2. LoRA 어댑터 로드")
print("3. 추론 수행")


---

## 📌 체크리스트

학습 완료 후 확인할 사항들:

- [ ] 데이터셋이 올바르게 로드되고 전처리되었는가?
- [ ] 모델이 4-bit로 양자화되어 메모리 효율적으로 로드되었는가?
- [ ] LoRA 설정이 적절한가? (rank, alpha, target_modules)
- [ ] 학습이 정상적으로 완료되었는가?
- [ ] Loss가 감소하는 추세를 보이는가?
- [ ] Overfitting 징후는 없는가? (train loss << eval loss)
- [ ] 생성된 답변이 의미 있고 도메인에 적합한가?
- [ ] 모델과 어댑터가 올바르게 저장되었는가?

### 트러블슈팅

**Q1: CUDA Out of Memory 에러가 발생합니다.**
- A: `per_device_train_batch_size`를 줄이거나 `gradient_accumulation_steps`를 늘리세요.
- A: `max_seq_length`를 512로 줄여보세요.

**Q2: 학습 속도가 너무 느립니다.**
- A: `num_train_epochs`를 줄이거나 데이터셋을 샘플링하세요.
- A: `gradient_checkpointing=False`로 설정하면 속도가 빨라지지만 메모리를 더 사용합니다.

**Q3: 모델이 이상한 답변을 생성합니다.**
- A: 더 많은 에포크로 학습하거나 learning rate를 조절해보세요.
- A: 데이터 품질을 확인하고 전처리를 개선하세요.

**Q4: LoRA 어댑터 파일이 너무 큽니다.**
- A: LoRA rank를 줄여보세요 (r=8 또는 r=4).
- A: target_modules를 줄여보세요.
